In [1]:
from cdo import Cdo
import xarray as xr
import pandas as pd
import glob
import os 

os.chdir('../..') # beginning of the repository

%load_ext autoreload
%autoreload 2

In [2]:
esm_log = pd.read_csv('data/logs/esm_log_tos.csv')

# SST/TOS remapping

In [3]:
source_path = 'data/reanalysis/sst.mnmean.89x180.nc'

cdo = Cdo()
SST_obs = xr.open_dataset(source_path)
nlat = SST_obs.lat.size
nlon = SST_obs.lon.size

try:
    grid_path = f'data/grids/sst_obs_{nlat}x{nlon}.txt'
    grid_info = cdo.griddes(input = source_path)
    # Save the string to a file or process it
    with open(grid_path, 'w') as f:
        for line in grid_info:
            f.write(line + '\n')
except Exception as e:
    print(f"Error: {e}")

nlat, nlon

(89, 180)

In [4]:
esm_subset = esm_log[(esm_log['Variable ID'] == 'tos') & (esm_log['Lat'] >= nlat) & (esm_log['Lon'] >= nlon)]
esm_subset.head()

,ESM,Experiment ID,Variable ID,Member ID,Lat,Lon,Grid Points
0,ACCESS-CM2,historical,tos,r1i1p1f1,108000,108000,11664000000
1,ACCESS-ESM1-5,historical,tos,r1i1p1f1,108000,108000,11664000000
2,AWI-CM-1-1-MR,historical,tos,r1i1p1f1,830305,830305,689406393025
3,AWI-ESM-1-1-LR,historical,tos,r1i1p1f1,126859,126859,16093205881
4,BCC-CSM2-MR,historical,tos,r1i1p1f1,83520,83520,6975590400


In [ ]:
var = 'tos'
target_grid = 'data/grids/sst_obs_89x180.txt'
output_path_base = f'data/CMIP6_monthly_data/remapped/{var}'
os.makedirs(output_path_base, exist_ok=True)

for idx, esm_info in esm_subset.iterrows():
    #print(esm_info['ESM'])
    print(f'Processing : {esm_info['ESM']}')
    
    #if esm_info['ESM'] == 'MPI-ESM1-2-HR':
    #    print(glob.glob(f'data/CMIP6_monthly_data/r1i1p1f1/{var}/historical/{esm_info['ESM']}*_historical_r1i1p1f1_{var}.nc'))

    input_path = glob.glob(f'data/CMIP6_monthly_data/r1i1p1f1/{var}/historical/{esm_info['ESM']}*_historical_r1i1p1f1_{var}.nc')
    output_path = os.path.join(output_path_base, f'{esm_info['ESM']}_{var}_{nlat}x{nlon}_remapped.nc')

    if os.path.exists(os.path.join(output_path)):
        print(f'    File {os.path.basename(output_path)} already exists ...')
        pass
    elif len(input_path) == 0:
        print(f'    File {os.path.basename(output_path)} not found ...')
        continue
    else:
        try:
            #remapped_data = cdo.remapcon(target_grid, input=input_path[0], output=output_path) #, options = '-f nc')
            remapped_data = cdo.remapbil(target_grid, input=input_path[0], output=output_path) #, options = '-f nc')
        except:
            print(f'    Remapping failed for {esm_info['ESM']}')
            #ValueError(f'Unable to remap {os.path.basename(input_path)} ...')
            continue

Processing : ACCESS-CM2
    File ACCESS-CM2_tos_89x180_remapped.nc already exists ...
Processing : ACCESS-ESM1-5
    File ACCESS-ESM1-5_tos_89x180_remapped.nc already exists ...
Processing : AWI-CM-1-1-MR
    File AWI-CM-1-1-MR_tos_89x180_remapped.nc already exists ...
Processing : AWI-ESM-1-1-LR
    File AWI-ESM-1-1-LR_tos_89x180_remapped.nc already exists ...
Processing : BCC-CSM2-MR
    File BCC-CSM2-MR_tos_89x180_remapped.nc already exists ...
Processing : BCC-ESM1
    File BCC-ESM1_tos_89x180_remapped.nc already exists ...
Processing : CAMS-CSM1-0
    File CAMS-CSM1-0_tos_89x180_remapped.nc already exists ...
Processing : CESM2
    File CESM2_tos_89x180_remapped.nc already exists ...
Processing : CESM2
    File CESM2_tos_89x180_remapped.nc already exists ...
Processing : CESM2-FV2
    File CESM2-FV2_tos_89x180_remapped.nc already exists ...
Processing : CESM2-FV2
    File CESM2-FV2_tos_89x180_remapped.nc already exists ...
Processing : CESM2-WACCM
    File CESM2-WACCM_tos_89x180_r